In [1]:
import pandas as pd
from pathlib import Path

In [9]:
df = pd.read_csv("../data/raw/oferta_gastronomica.csv", sep=";", encoding="latin-1")
df.head()


,long,lat,id,nombre,categoria,cocina,ambientacion,telefono,mail,horario,calle_nombre,calle_altura,calle_cruce,direccion_completa,barrio,comuna,codigo_postal,codigo_postal_argentino
0,"-58,36330675","-34,62296347",1602,GUANTANAMERA,RESTAURANTE,CUBANA,MODERNO,4314 0770,NaN,NaN,MOREAU DE JUSTO ALICIA,2092,NaN,"MOREAU DE JUSTO, ALICIA AV. 2092",Puerto Madero,1,1107.0,C1107AFP
1,"-58,39232963","-34,62155353",676,1843 VINACOTECA,VINERIA,NaN,NaN,4779 0806,NaN,lunes a domingo de 8 a 00,HUMBERTO 1º,1843,NaN,HUMBERTO 1º 1843,San Cristobal,3,NaN,NaN
2,"-58,45354227","-34,56031135",487,2080 EXPRESS,RESTAURANTE,TRADICIONAL,CLASICO PORTEÑO,4788 4500,NaN,lunes a domingo de 8 a 00,JURAMENTO,2080,NaN,JURAMENTO AV. 2080,Belgrano,13,1428.0,C1428DNH
3,"-58,47549105","-34,6770507",2737,22,CAFE,MINUTAS,MODERNO,NaN,NaN,NaN,MURGUIONDO,4188,NaN,MURGUIONDO 4188,Villa Lugano,8,1439.0,C1439FTB
4,"-58,4254372","-34,58569501",2,4 EN 2 I,RESTAURANTE,NaN,NaN,4832 9510,NaN,NaN,BORGES JORGE LUIS,2095,NaN,"BORGES, JORGE LUIS 2095",Palermo,14,1425.0,C1425FFA


In [10]:
print(f"filas: {df.shape[0]}, columnas: {df.shape[1]}")

df.describe().round(2)

filas: 2823, columnas: 18


,id,calle_altura,comuna,codigo_postal
count,2823.00,2823.00,2823.00,2766.00
mean,1427.92,2145.93,6.02,1239.19
std,822.46,1851.60,5.29,173.24
min,1.00,0.00,1.00,1001.00
25%,718.50,762.50,1.00,1057.00
50%,1427.00,1606.00,3.00,1207.00
75%,2140.50,3001.00,12.00,1424.00
max,2854.00,11700.00,15.00,1440.00


In [ ]:
#Comienzo inspeccionando nulos y duplicados

#Calculo la cantidad de valores nulos  en cada columna
nulos =  df.isnull().sum()
print(f'Valores nulos por columna: {nulos}')


Valores nulos por columna: long                          0
lat                           0
id                            0
nombre                        0
categoria                   116
cocina                      359
ambientacion                559
telefono                    520
mail                       2351
horario                    2098
calle_nombre                  0
calle_altura                  0
calle_cruce                2817
direccion_completa            1
barrio                        0
comuna                        0
codigo_postal                57
codigo_postal_argentino      66
dtype: int64


In [6]:
#Calculo la cantidad de filas duplcadas
duplicados = df.duplicated().sum()
print(f'Cantidad de filas duplicadas: {duplicados}')

Cantidad de filas duplicadas: 0


In [7]:
#Comienzo con la estandarizacion y limpieza
#realizo copia de seguridad
df_limpio = df.copy()


In [ ]:
#trato valores nulos y estandarizo colmnas de textp
# Relleno los NaN con 'No especificado' para no perder registros 
columnas_texto = ['categoria', 'cocina', 'ambientacion', 'barrio']
for col in columnas_texto:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].fillna('No especificado')
        # Estandarizamos el texto: pasa a string, se quitan espacios extra y se aplico formato título
        df_limpio[col] = df_limpio[col].astype(str).str.strip().str.title()

In [ ]:
#agrupacion por comuna y barrio. comuna (visión macro y logística) y barrio (detalle micro local 
# para detectar baches de oferta y saturación comercial con precisión)
df_comuna = df_limpio.groupby(['comuna', 'barrio']).agg(
    total_establecimientos=('nombre', 'count'),
    categorias_unicas=('categoria', 'nunique')
).reset_index()

In [15]:
#ordenado de mayor a menor 
df_comuna = df_comuna.sort_values(by='total_establecimientos', ascending=False)

In [16]:
print('Comunas y barrios con mayor densidad: ')
print(df_comuna.head())

Comunas y barrios con mayor densidad: 
   comuna       barrio  total_establecimientos  categorias_unicas
23     14      Palermo                     467                  9
4       1  San Nicolas                     454                  7
29      2     Recoleta                     327                  8
30      3    Balvanera                     215                  8
1       1    Monserrat                     149                  5


In [ ]:
# Exportar los datasets limpios a la carpeta data/processed/
df_limpio.to_csv("../data/processed/oferta_gastronomica_limpia.csv", index=False, encoding="utf-8")
df_comuna.to_csv("../data/processed/densidad_comercial_por_comuna.csv", index=False, encoding="utf-8")
